# MCP Agent POC

Minimal LangChain / LangGraph proof of concept that loads tools from the asyncroscopy MCP server and asks the model to use one or two of them.

## Prereqs

Start the Tango stack and MCP server first:

```bash
uv run startup_scripts/run_servers.py
uv run startup_scripts/run_mcp.py --yaml configs/mcp.yaml
```

If the agent packages are missing from your environment, install them into the same project environment first:

```bash
uv add langchain langgraph langchain-mcp-adapters
```

In [ ]:
import os
from pprint import pprint

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_mcp_adapters.client import MultiServerMCPClient

MCP_URL = os.getenv("ASYNCROSCOPY_MCP_URL", "http://127.0.0.1:8000/mcp")
MODEL_NAME = os.getenv("ASYNCROSCOPY_MODEL", "openai:gpt-4o-mini")

print("MCP_URL =", MCP_URL)
print("MODEL_NAME =", MODEL_NAME)

In [ ]:
client = MultiServerMCPClient(
    {
        "asyncroscopy": {
            "url": MCP_URL,
            "transport": "streamable_http",
        }
    }
)

tools = await client.get_tools()
print(f"Loaded {len(tools)} tool(s)")
print([tool.name for tool in tools[:10]])

In [ ]:
model = init_chat_model(MODEL_NAME)
agent = create_agent(model, tools)

def final_text(result):
    message = result["messages"][-1]
    return getattr(message, "content", message)

result = await agent.ainvoke({
    "messages": "List the available MCP devices or tools in one short sentence, then stop."
})
print(final_text(result))

In [ ]:
tool_names = [tool.name.lower() for tool in tools]
imageish_tools = [
    tool_name
    for tool_name in tool_names
    if any(token in tool_name for token in ("image", "acquire", "scan", "camera", "detector"))
]

if imageish_tools:
    prompt = (
        "Use exactly one MCP tool to get a single image or acquisition result. "
        f"Prefer one of these tools if it fits: {', '.join(imageish_tools)}."
    )
else:
    prompt = "No obvious image tool was exposed, so just report that and stop."

result = await agent.ainvoke({"messages": prompt})
print(final_text(result))